# Graph OLAP — End-to-End Demo
**Pipeline:** Starburst TPC-H → Export Worker → GCS Parquet → FalkorDB → Cypher

In [ ]:
import requests, json, time, subprocess

API = "http://graph-olap-control-plane:8080"
H = {"X-Username": "demo@example.com", "X-User-Role": "admin", "Content-Type": "application/json"}

def get(path): r = requests.get(f"{API}{path}", headers=H); r.raise_for_status(); return r.json()
def post(path, body): r = requests.post(f"{API}{path}", headers=H, json=body); r.raise_for_status(); return r.json()

print(requests.get(f"{API}/health").json())

## Step 1 — Create Mapping
A **mapping** defines your graph: which SQL rows become nodes, which joins become edges.

In [ ]:
mapping = post("/api/mappings", {
    "name": "TPC-H Supply Chain",
    "description": "Customer/Nation/SalesOrder graph from TPC-H tiny dataset",
    "node_definitions": [
        {
            "label": "Customer",
            "sql": "SELECT custkey, name, acctbal, mktsegment FROM tpch.tiny.customer",
            "primary_key": {"name": "custkey", "type": "INT64"},
            "properties": [
                {"name": "name",       "type": "STRING"},
                {"name": "acctbal",    "type": "DOUBLE"},
                {"name": "mktsegment", "type": "STRING"}
            ]
        },
        {
            "label": "Nation",
            "sql": "SELECT nationkey, name FROM tpch.tiny.nation",
            "primary_key": {"name": "nationkey", "type": "INT64"},
            "properties": [{"name": "name", "type": "STRING"}]
        },
        {
            "label": "SalesOrder",
            "sql": "SELECT orderkey, custkey, totalprice, orderstatus FROM tpch.tiny.orders",
            "primary_key": {"name": "orderkey", "type": "INT64"},
            "properties": [
                {"name": "custkey",    "type": "INT64"},
                {"name": "totalprice", "type": "DOUBLE"},
                {"name": "orderstatus","type": "STRING"}
            ]
        }
    ],
    "edge_definitions": [
        {
            "type": "BELONGS_TO",
            "from_node": "Customer", "to_node": "Nation",
            "sql": "SELECT custkey, nationkey FROM tpch.tiny.customer",
            "from_key": "custkey", "to_key": "nationkey", "properties": []
        },
        {
            "type": "PLACED",
            "from_node": "Customer", "to_node": "SalesOrder",
            "sql": "SELECT custkey, orderkey FROM tpch.tiny.orders",
            "from_key": "custkey", "to_key": "orderkey", "properties": []
        }
    ]
})

mapping_id = mapping["data"]["id"]
print(f"✅ Mapping created: id={mapping_id}  name={mapping['data']['name']}")
print(f"   Nodes: {[n['label'] for n in mapping['data']['node_definitions']]}")
print(f"   Edges: {[e['type'] for e in mapping['data']['edge_definitions']]}")

## Step 2 — Create Instance
An **instance** is a live graph database. Creating one triggers the full export pipeline.

> **Note:** After creation we patch the DB so the export worker uses the correct Starburst catalog.

In [ ]:
inst = post("/api/instances", {
    "mapping_id": mapping_id,
    "wrapper_type": "falkordb",
    "name": "TPC-H Demo",
    "ttl": "PT2H"
})

instance_id = inst["data"]["id"]
snapshot_id = inst["data"]["snapshot_id"]
print(f"✅ Instance created: id={instance_id}  snapshot_id={snapshot_id}")
print(f"   Status: {inst['data']['status']}")
print()

# DB patch: fix starburst_catalog from 'bigquery' -> 'tpch'
import psycopg2
conn = psycopg2.connect(host="postgres", port=5432, dbname="control_plane",
                        user="control_plane", password="control_plane")
conn.autocommit = True
with conn.cursor() as cur:
    cur.execute(
        "UPDATE export_jobs SET starburst_catalog = 'tpch' WHERE snapshot_id = %s AND starburst_catalog = 'bigquery'",
        (snapshot_id,)
    )
    print(f"✅ DB patch: updated {cur.rowcount} export jobs -> starburst_catalog='tpch'")
conn.close()

## Step 3 — Monitor Export Progress
The export worker is now:
1. Querying Starburst Galaxy for Customer, Nation, Order, BELONGS_TO, PLACED
2. Writing Parquet files to GCS (fake-gcs-local)
3. Once all jobs complete → FalkorDB pod will start

In [ ]:
import psycopg2

def db_query(sql, params=None):
    conn = psycopg2.connect(host="postgres", port=5432, dbname="control_plane",
                            user="control_plane", password="control_plane")
    with conn.cursor() as cur:
        cur.execute(sql, params)
        rows = cur.fetchall()
    conn.close()
    return rows

print("Polling... (Starburst queries take ~30-90s each)\n")

for i in range(60):
    time.sleep(10)

    jobs = db_query(
        "SELECT entity_name, status FROM export_jobs WHERE snapshot_id = %s ORDER BY id",
        (snapshot_id,)
    )
    jobs_str = "  ".join(f"{name}:{st}" for name, st in jobs)

    try:
        inst_data = get(f"/api/instances/{instance_id}")["data"]
        status = inst_data["status"]
    except:
        status = "unknown"

    print(f"[{(i+1)*10:3d}s] instance={status:25s} | {jobs_str}")

    if status in ["running", "ready"]:
        print(f"\n🎉 Instance is {status}! FalkorDB is ready.")
        print(f"   Pod: {inst_data.get('pod_name')}")
        break
    elif status == "failed":
        print(f"\n❌ Failed: {inst_data.get('error_message')}")
        break

## Step 4 — Connect to FalkorDB and Run Cypher Queries

In [ ]:
inst_data = get(f"/api/instances/{instance_id}")["data"]
pod_name = inst_data.get("pod_name", "")
print("Pod:", pod_name)

# The wrapper service name = pod name (full UUID)
# It exposes a REST API on port 8000 with POST /query for Cypher
WRAPPER = f"http://{pod_name}:8000"

def cypher(query, params=None):
    body = {"query": query}
    if params:
        body["parameters"] = params
    r = requests.post(f"{WRAPPER}/query", headers={"Content-Type": "application/json"}, json=body)
    r.raise_for_status()
    return r.json()

# Test connection
result = cypher("MATCH (n) RETURN labels(n)[0] as label, count(n) as cnt ORDER BY label")
print("\n=== Graph loaded — Node counts ===")
for row in result["rows"]:
    print(f"  {str(row[0]):20s}: {row[1]:,}")

In [ ]:
result = cypher("""
    MATCH (c:Customer)-[:BELONGS_TO]->(n:Nation)
    RETURN n.name AS nation, count(c) AS customers
    ORDER BY customers DESC
    LIMIT 10
""")
print("=== Customers per Nation ===")
print(f"  {'Nation':30s}  Customers")
print("  " + "-"*40)
for row in result["rows"]:
    print(f"  {str(row[0]):30s}  {row[1]}")

In [ ]:
result = cypher("""
    MATCH (c:Customer)-[:PLACED]->(o:SalesOrder)
    RETURN c.name AS customer, sum(o.totalprice) AS total_spent
    ORDER BY total_spent DESC
    LIMIT 10
""")
print("=== Top 10 Customers by Spend ===")
print(f"  {'Customer':40s}  Total Spent")
print("  " + "-"*55)
for row in result["rows"]:
    print(f"  {str(row[0]):40s}  ${row[1]:,.2f}")

In [ ]:
result = cypher("""
    MATCH (n:Nation)<-[:BELONGS_TO]-(c:Customer)-[:PLACED]->(o:SalesOrder)
    WHERE n.name = 'BRAZIL'
    RETURN c.name AS customer, count(o) AS orders, sum(o.totalprice) AS total
    ORDER BY total DESC
    LIMIT 5
""")
print("=== Brazilian Customers ===")
for row in result["rows"]:
    print(f"  {str(row[0]):35s}  {row[1]} orders  ${row[2]:,.2f}")

In [ ]:
result = cypher("""
    MATCH (c:Customer)-[:PLACED]->(o:SalesOrder)
    WITH c.mktsegment AS segment, count(o) AS orders, sum(o.totalprice) AS revenue
    RETURN segment, orders, revenue
    ORDER BY revenue DESC
""")
print("=== Revenue by Market Segment ===")
print(f"  {'Segment':15s}  {'Orders':>8s}  Revenue")
print("  " + "-"*45)
for row in result["rows"]:
    print(f"  {str(row[0]):15s}  {row[1]:>8,}  ${row[2]:,.2f}")

## Step 5 — Visualize the Graph

We'll sample a small subgraph (5 nations + their customers + a few orders) and render it as an **interactive visual graph** using PyVis.

In [ ]:
!pip install pyvis -q
from pyvis.network import Network
import IPython.display as display

# --- Fetch sample data: 5 nations + up to 5 customers each + up to 3 orders per customer ---

nations = cypher("MATCH (n:Nation) RETURN n.nationkey, n.name LIMIT 5")["rows"]

net = Network(height="600px", width="100%", bgcolor="#1a1a2e", font_color="white", notebook=True)
net.set_options("""
{
  "physics": {"stabilization": {"iterations": 200}},
  "edges": {"arrows": {"to": {"enabled": true}}, "color": {"inherit": false}},
  "interaction": {"hover": true}
}
""")

nation_color   = "#e94560"   # red
customer_color = "#0f3460"   # dark blue  (border white)
order_color    = "#533483"   # purple

added_nodes = set()

for nkey, nname in nations:
    nid = f"nation_{nkey}"
    net.add_node(nid, label=nname, color=nation_color, size=30, title=f"Nation: {nname}", shape="star")
    added_nodes.add(nid)

    custs = cypher("""
        MATCH (c:Customer)-[:BELONGS_TO]->(n:Nation)
        WHERE n.nationkey = $nk
        RETURN c.custkey, c.name, c.mktsegment
        LIMIT 5
    """, {"nk": nkey})["rows"]

    for ckey, cname, seg in custs:
        cid = f"cust_{ckey}"
        if cid not in added_nodes:
            net.add_node(cid, label=cname[:15], color=customer_color,
                         size=18, title=f"{cname}\nSegment: {seg}", shape="dot",
                         borderWidth=2, borderColor="white")
            added_nodes.add(cid)
        net.add_edge(cid, nid, color="#aaaaaa", label="BELONGS_TO", title="BELONGS_TO")

        orders = cypher("""
            MATCH (c:Customer)-[:PLACED]->(o:SalesOrder)
            WHERE c.custkey = $ck
            RETURN o.orderkey, o.totalprice, o.orderstatus
            LIMIT 3
        """, {"ck": ckey})["rows"]

        for okey, price, status in orders:
            oid = f"order_{okey}"
            if oid not in added_nodes:
                net.add_node(oid, label=f"${price:,.0f}", color=order_color,
                             size=10, title=f"Order #{okey}\nStatus: {status}\nTotal: ${price:,.2f}",
                             shape="square")
                added_nodes.add(oid)
            net.add_edge(cid, oid, color="#7a5c9e", label="PLACED", title="PLACED")

print(f"Graph: {len(net.nodes)} nodes, {len(net.edges)} edges")
net.show("graph.html")
display.HTML("graph.html")